learning-vector-quantization steps

In [33]:
import numpy as np
import random

In [25]:
def euclidean_distance(x1, x2):
    x1_np = np.array(x1, dtype=np.float64)
    x2_np = np.array(x2, dtype=np.float64)
    distance = 0.0
    distance += np.sum((x1_np - x2_np) ** 2)
    return np.sqrt(distance)

In [26]:
# --- Named test cases with known expected distances ---
euclidean_distance_test_cases = [
    # (point_a, point_b, expected_distance)
    ((0, 0), (3, 4),    5.0),          # 3-4-5 triangle
    ((0, 0), (0, 0),    0.0),          # identical points
    ((1, 1), (4, 5),    5.0),          # shifted 3-4-5
    ((0, 0), (1, 0),    1.0),          # unit distance along x
    ((2, 3), (2, 3),    0.0),          # identical non-origin points
    ((-1, -1), (2, 3),  5.0),          # negative to positive
    ((0, 0), (1, 1),    np.sqrt(2)),   # diagonal unit square
]
for a, b, expected in euclidean_distance_test_cases:
    result = euclidean_distance(a, b)
    assert abs(result - expected) < 1e-9, f"Failed: dist({a}, {b}) = {result}, expected {expected}"

In [ ]:
import time
import numpy as np
from scipy.spatial.distance import cdist

# --- Dataset ---
rng = np.random.default_rng(seed=42)
points = rng.uniform(-10, 10, size=(100, 128))

# --- Correctness: compare every pairwise distance against scipy's reference ---
expected_matrix = cdist(points, points, metric="euclidean")

computed_matrix = np.array([
    [euclidean_distance(points[i], points[j]) for j in range(len(points))]
    for i in range(len(points))
])

assert np.allclose(computed_matrix, expected_matrix, atol=1e-9), "High-dim correctness check failed"
print("✓ Correctness passed (100x100 pairwise, 128-dim)")

# --- Performance: clock your implementation vs scipy ---
N_RUNS = 10

start = time.perf_counter()
for _ in range(N_RUNS):
    _ = np.array([
        [euclidean_distance(points[i], points[j]) for j in range(len(points))]
        for i in range(len(points))
    ])
elapsed_yours = (time.perf_counter() - start) / N_RUNS

start = time.perf_counter()
for _ in range(N_RUNS):
    _ = cdist(points, points, metric="euclidean")
elapsed_scipy = (time.perf_counter() - start) / N_RUNS

print(f"Your implementation : {elapsed_yours * 1000:.3f} ms/run")
print(f"scipy cdist         : {elapsed_scipy * 1000:.3f} ms/run")
print(f"Slowdown            : {elapsed_yours / elapsed_scipy:.1f}x")

# numpy for-loop
# ✓ Correctness passed (100x100 pairwise, 128-dim)
# Your implementation : 2141.395 ms/run
# scipy cdist         : 0.270 ms/run
# Slowdown            : 7936.0x

# numpy vectorized
# ✓ Correctness passed (100x100 pairwise, 128-dim)
# Your implementation : 27.265 ms/run
# scipy cdist         : 0.294 ms/run
# Slowdown            : 92.9x

✓ Correctness passed (100x100 pairwise, 128-dim)
Your implementation : 27.265 ms/run
scipy cdist         : 0.294 ms/run
Slowdown            : 92.9x


In [30]:
def best_matching_unit(codebooks, test_row):
    distance_list = []
    for codebook in codebooks:
        distance = euclidean_distance(codebook, test_row)
        distance_list.append((codebook, distance))
    distance_list.sort(key=lambda tup: tup[1])
    return distance_list[0][0]

In [31]:
# --- Named test cases with known expected BMU ---
bmu_test_cases = [
    {
        "codebooks": [
            [1.0, 1.0],   # ← closest to test_row
            [9.0, 9.0],
            [5.0, 5.0],
        ],
        "test_row": [1.5, 1.5],
        "expected_bmu": [1.0, 1.0],
        "description": "clear winner, 2D"
    },
    {
        "codebooks": [
            [0.0, 0.0],
            [3.0, 4.0],   # ← closest (distance = 0, exact match)
            [10.0, 10.0],
        ],
        "test_row": [3.0, 4.0],
        "expected_bmu": [3.0, 4.0],
        "description": "exact match in codebook"
    },
    {
        "codebooks": [
            [0.0, 0.0, 0.0],
            [1.0, 2.0, 2.0],  # ← closest (distance = 3.0 from origin)
            [9.0, 9.0, 9.0],
        ],
        "test_row": [0.5, 1.5, 1.5],
        "expected_bmu": [1.0, 2.0, 2.0],
        "description": "3D points"
    },
    {
        "codebooks": [
            [-5.0, -5.0],  # ← closest to a negative test_row
            [5.0, 5.0],
            [0.0, 0.0],
        ],
        "test_row": [-4.0, -4.0],
        "expected_bmu": [-5.0, -5.0],
        "description": "negative coordinates"
    },
    {
        "codebooks": [
            [1.0, 0.0],
        ],
        "test_row": [99.0, 99.0],
        "expected_bmu": [1.0, 0.0],
        "description": "single codebook, must always be returned"
    },
]

In [32]:
for case in bmu_test_cases:
    result = best_matching_unit(case["codebooks"], case["test_row"])
    assert result == case["expected_bmu"], (
        f"FAILED [{case['description']}]: got {result}, expected {case['expected_bmu']}"
    )
    print(f"✓ {case['description']}")

✓ clear winner, 2D
✓ exact match in codebook
✓ 3D points
✓ negative coordinates
✓ single codebook, must always be returned


In [40]:
def random_codebook(train):
    n_records = len(train)
    n_features = len(train[0])
    codebook = [train[random.randrange(n_records)][i] for i in range(n_features)]
    return codebook

In [43]:
def train_codebooks(train, n_codebooks, l_rate=0.3, epochs=100):
    codebooks = [random_codebook(train) for i in range(n_codebooks)]
    for epoch in range(epochs):
        rate = l_rate * (1.0 - (epoch /float(epochs)))
        sum_error = 0.0
        for row in train:
            bmu = best_matching_unit(codebooks, test_row=row)
            for i in range(len(row)-1):
                error = row[i] - bmu[i]
                sum_error += error ** 2
                if bmu[-1] == row[-1]:
                    bmu[i] += rate * error
                else:
                    bmu[i] -= rate * error
        print(f"> epoch={epoch}, lrate={rate}, error={sum_error}")
    return codebooks

In [44]:
# Test the training function
random.seed(1)
dataset = [[2.7810836,2.550537003,0],
	[1.465489372,2.362125076,0],
	[3.396561688,4.400293529,0],
	[1.38807019,1.850220317,0],
	[3.06407232,3.005305973,0],
	[7.627531214,2.759262235,1],
	[5.332441248,2.088626775,1],
	[6.922596716,1.77106367,1],
	[8.675418651,-0.242068655,1],
	[7.673756466,3.508563011,1]]
learn_rate = 0.3
n_epochs = 10
n_codebooks = 2
codebooks = train_codebooks(dataset, n_codebooks, learn_rate, n_epochs)
print('Codebooks: %s' % codebooks)

> epoch=0, lrate=0.3, error=43.26953154012358
> epoch=1, lrate=0.27, error=30.40349857541859
> epoch=2, lrate=0.24, error=27.14561929988356
> epoch=3, lrate=0.21, error=26.300754252623506
> epoch=4, lrate=0.18, error=25.537410974881567
> epoch=5, lrate=0.15, error=24.789043079985422
> epoch=6, lrate=0.12, error=24.05804188553826
> epoch=7, lrate=0.09000000000000001, error=23.346138823666227
> epoch=8, lrate=0.059999999999999984, error=22.65400627230968
> epoch=9, lrate=0.029999999999999992, error=21.982079634512548
Codebooks: [[2.432316086217663, 2.839821664184211, 0], [7.319592257892681, 1.97013382654341, 1]]
